# 03. Avaluació del Model ARIMA (Filtre de Kalman i Sliding Window)

En aquest quadern implemente l'algoritme de predicció basat en el model Autoregressiu Integrat de Mitjana Mòbil (ARIMA). Establisc l'ordre (1, 1, 0) com a base estructural i configure l'entorn experimental per a realitzar una avaluació contínua mitjançant una finestra lliscant de 12 hores totals (11 hores de *warm-up* i 1 hora de predicció).

In [16]:
# Importació de llibreries necessàries
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings, time
warnings.filterwarnings('ignore')

# ============================================================================
# PARÀMETRES CONFIGURABLES DEL DISSENY EXPERIMENTAL
# ============================================================================
MOSTRES_WARMUP = 132  # 11 hores de context obligatori (11h × 12 mostres/h)
MOSTRES_PRED   = 12   # 1 hora d'horitzó de predicció cap al futur
SALT_FINESTRA  = 1    # Pas del desplaçament (1 mostra = 5 minuts)

### 1. Càrrega de Dades i Validació de Llindars

Carregue els conjunts de dades prèviament netejats. Abans de procedir, incorpore una asserció de seguretat (assert) per a garantir matemàticament que tots els segments continus posseeixen un mínim de 144 mostres (12 hores). Aquest pas és crític per a assegurar que el model disposa de dades suficients per a conformar, com a mínim, una finestra de predicció completa.

In [17]:
# Càrrega dels datasets
df_train = pd.read_csv("dades_preprocessades/dataset_entrenament_net.csv")
df_test = pd.read_csv("dades_preprocessades/dataset_test_net.csv")

# Verificació de seguretat: el preprocessament ha d'haver aplicat el filtre de 12h
min_mostres_train = df_train.groupby(['pacient_id', 'segment_num']).size().min()
min_mostres_test = df_test.groupby(['pacient_id', 'segment_num']).size().min()

assert min_mostres_train >= 144, f"Error: Hi ha segments curts en Train ({min_mostres_train} mostres)."
assert min_mostres_test >= 144, f"Error: Hi ha segments curts en Test ({min_mostres_test} mostres)."
print("✅ Validació correcta: Tots els segments compleixen el llindar mínim de 12h (144 mostres).")

✅ Validació correcta: Tots els segments compleixen el llindar mínim de 12h (144 mostres).


### 2. Calibració de Coeficients ARIMA

Tot i utilitzar una estructura global `ARIMA(1, 1, 0)`, ajuste els coeficients (el paràmetre autoregressiu i la variància) de forma individualitzada per a cada pacient. Per a fer-ho, extraig el segment continu més extens del conjunt d'entrenament de cada subjecte i realitze l'ajust de paràmetres, desant els resultats en un diccionari per a la seua posterior reutilització.

In [18]:
print("\n1. Calibrant coeficients ARIMA(1,1,0) per a cada pacient...")
dic_coefs_pacient = {}
resultats_coefs = []

# Identifique el segment més llarg de cada pacient al conjunt de train
mides_segments = df_train.groupby(['pacient_id', 'segment_num']).size().reset_index(name='longitud')
idx_max = mides_segments.groupby('pacient_id')['longitud'].idxmax()
segments_mes_llargs = mides_segments.loc[idx_max]

# Ajuste el model i guarde els paràmetres individualitzats
for _, row in segments_mes_llargs.iterrows():
    pacient = row['pacient_id']
    seg_num = row['segment_num']
    
    df_seg = df_train[(df_train['pacient_id'] == pacient) & (df_train['segment_num'] == seg_num)]
    y_train = df_seg['glucose_imputed'].values
    
    model = ARIMA(y_train, order=(1, 1, 0))
    model_fit = model.fit()
    
    # Emmagatzeme l'objecte de paràmetres amb precisió completa per a la fase de test
    dic_coefs_pacient[pacient] = model_fit.params
    
    resultats_coefs.append({
        'Pacient': pacient,
        'Coeficient_AR(1)': round(model_fit.params[0], 4),
        'Variància_(sigma2)': round(model_fit.params[1], 4)
    })

df_coefs_final = pd.DataFrame(resultats_coefs)
print("Calibració finalitzada i coeficients emmagatzemats.")


1. Calibrant coeficients ARIMA(1,1,0) per a cada pacient...
Calibració finalitzada i coeficients emmagatzemats.


### 3. Avaluació en Test (Finestra Lliscant)

Avalue l'algoritme sobre el conjunt de test aplicant una tècnica de validació contínua. Implemente un bucle que desplaça la finestra de 5 en 5 minuts. En cada pas, utilitze el mètode `.filter()` per a actualitzar els estats del filtre de Kalman amb les 11 hores prèvies (*warm-up*), propagant els paràmetres prèviament congelats sense necessitat de reentrenar. Seguidament, projecte la predicció a 1 hora vista.

In [19]:
print("\n2. Avaluant ARIMA(1,1,0) en Test amb finestra lliscant...")
start_time = time.time()
prediccions_test = []

for pacient in df_test['pacient_id'].unique():
    df_pacient = df_test[df_test['pacient_id'] == pacient]
    params_pacient = dic_coefs_pacient[pacient]
    
    for seg_num, df_seg in df_pacient.groupby('segment_num'):
        df_seg = df_seg.sort_values('timestamp').reset_index(drop=True)
        y_test = df_seg['glucose_imputed'].values
        n_mostres = len(y_test)
        
        # Bucle de la finestra lliscant sense deformar el bloc temporal
        for i in range(0, n_mostres - MOSTRES_WARMUP - MOSTRES_PRED + 1, SALT_FINESTRA):
            y_warmup = y_test[i : i + MOSTRES_WARMUP]
            y_real_futur = y_test[i + MOSTRES_WARMUP : i + MOSTRES_WARMUP + MOSTRES_PRED]
            
            # Actualització d'estats interns amb els paràmetres reutilitzats
            model_eval = ARIMA(y_warmup, order=(1, 1, 0))
            res_eval = model_eval.filter(params_pacient)
            
            # Predicció dels pròxims 12 passos (60 minuts)
            preds = res_eval.forecast(steps=MOSTRES_PRED)

            # Timestamp de l'última observació del context (moment en què es fa la predicció)
            timestamp_pred = df_seg['timestamp'].iloc[i + MOSTRES_WARMUP - 1]
            
            prediccions_test.append({
                'Pacient': pacient,
                'Timestamp_Pred': timestamp_pred,
                'Real_15m': y_real_futur[2],  'Pred_15m': preds.iloc[2] if hasattr(preds, 'iloc') else preds[2],
                'Real_30m': y_real_futur[5],  'Pred_30m': preds.iloc[5] if hasattr(preds, 'iloc') else preds[5],
                'Real_45m': y_real_futur[8],  'Pred_45m': preds.iloc[8] if hasattr(preds, 'iloc') else preds[8],
                'Real_60m': y_real_futur[11], 'Pred_60m': preds.iloc[11] if hasattr(preds, 'iloc') else preds[11]
            })

df_res = pd.DataFrame(prediccions_test)
temps_exec = time.time() - start_time
print(f"   Avaluació completada en {temps_exec/60:.2f} minuts. Finestres processades: {len(df_res)}")


2. Avaluant ARIMA(1,1,0) en Test amb finestra lliscant...
   Avaluació completada en 0.67 minuts. Finestres processades: 11394


### 4. Càlcul de Mètriques i Exportació

S'extrauen i es presenten els resultats globals del model. Es calculen l'Error Quadràtic Mitjà (RMSE), l'Error Absolut Mitjà (MAE) i el MARD (Error Relatiu Absolut Mitjà) amb protecció matemàtica per als horitzons clínics de 15, 30, 45 i 60 minuts. Les mètriques i els coeficients s'exporten per a la memòria final.

In [20]:
def calcular_metriques_segures(real, pred):
    rmse = np.sqrt(mean_squared_error(real, pred))
    mae = mean_absolute_error(real, pred)
    
    # Protecció davant d'una impossible divisió per zero en glucosa
    real_segur = np.where(real == 0, np.nan, real)
    mard = np.nanmean(np.abs((real - pred) / real_segur)) * 100
    return rmse, mae, mard

horitzons = ['15m', '30m', '45m', '60m']
taula_metriques = []

for h in horitzons:
    rmse, mae, mard = calcular_metriques_segures(df_res[f'Real_{h}'], df_res[f'Pred_{h}'])
    taula_metriques.append({'Horitzó': h, 'RMSE (mg/dL)': round(rmse,2), 
                            'MAE (mg/dL)': round(mae,2), 'MARD (%)': round(mard,2)})

df_metriques = pd.DataFrame(taula_metriques)

print("\n" + "="*60)
print(" TAULA 1: RESULTATS GLOBALS EN TEST (ARIMA 1,1,0)")
print("="*60)
print(df_metriques.to_string(index=False))

print("\n" + "="*60)
print(" TAULA 2: COEFICIENTS INDIVIDUALITZATS PER PACIENT")
print("="*60)
print(df_coefs_final.to_string(index=False))

# Guardem la versió definitiva
df_res.to_csv("resultats_metrics/prediccions_arima_test.csv", index=False)
df_metriques.to_csv("resultats_metrics/arima_metriques_test.csv", index=False)
df_coefs_final.to_csv("resultats_metrics/arima_coeficients_pacients.csv", index=False)
print("\nExportació completada.")


 TAULA 1: RESULTATS GLOBALS EN TEST (ARIMA 1,1,0)
Horitzó  RMSE (mg/dL)  MAE (mg/dL)  MARD (%)
    15m         12.69         7.86      5.06
    30m         21.63        14.55      9.35
    45m         28.98        20.41     13.14
    60m         35.21        25.41     16.50

 TAULA 2: COEFICIENTS INDIVIDUALITZATS PER PACIENT
 Pacient  Coeficient_AR(1)  Variància_(sigma2)
     559            0.6104             26.5439
     563            0.5928             13.5789
     570            0.6597             10.9817
     575            0.7740              9.4013
     588            0.6726             12.1637
     591            0.7645              9.7324

Exportació completada.


### 5. Avaluació Clínica: Detecció d'Hipoglucèmies

Per a quantificar la utilitat clínica del model ARIMA, traduïsc els resultats continus a un problema de classificació binària. Establisc el llindar de risc en $\le 70$ mg/dL per a detectar esdeveniments d'hipoglucèmia. Mitjançant l'anàlisi de la matriu de confusió en cada finestra de predicció, calcule mètriques clau com la Sensibilitat (*Recall*), la Precisió i l'F1-Score. Finalment, exporte aquesta taula per a avaluar la capacitat del model a l'hora d'evitar falsos negatius crítics.

In [21]:
LLINDAR_HIPO = 70.0
resultats_hipo = []

for h in ['15m', '30m', '45m', '60m']:
    # Extracció de matrius per a l'horitzó corresponent
    y_true = df_res[f'Real_{h}'].values
    y_pred = df_res[f'Pred_{h}'].values

    # Binarització de les prediccions segons el llindar clínic
    real_hipo = (y_true <= LLINDAR_HIPO)
    pred_hipo = (y_pred <= LLINDAR_HIPO)

    # Càlcul de la Matriu de Confusió
    tp = np.sum(real_hipo & pred_hipo)        # Veritables Positius
    fn = np.sum(real_hipo & ~pred_hipo)       # Falsos Negatius
    fp = np.sum(~real_hipo & pred_hipo)       # Falsos Positius
    tn = np.sum(~real_hipo & ~pred_hipo)      # Veritables Negatius

    # Càlcul de mètriques amb prevenció de divisió per zero
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Acumulació de resultats
    resultats_hipo.append({
        'Horitzó'                    : h,
        'Casos_Hipo_Reals'           : int(np.sum(real_hipo)),
        'TP (Detectats)'             : int(tp),
        'FN (Omesos)'                : int(fn),
        'FP (Falses Alarmes)'        : int(fp),
        'Sensibilitat / Recall (%)'  : round(recall * 100, 2),
        'Precisió (%)'               : round(precision * 100, 2),
        'F1-Score (%)'               : round(f1 * 100, 2)
    })

df_hipo = pd.DataFrame(resultats_hipo)

print("\n" + "="*80)
print(" TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL ARIMA")
print("="*80)
print(df_hipo.to_string(index=False))

# Exportació dels resultats de classificació específics del model ARIMA
df_hipo.to_csv("resultats_metrics/arima_hipoglucemies.csv", index=False)
print("\nExportació de mètriques clíniques completada.")


 TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL ARIMA
Horitzó  Casos_Hipo_Reals  TP (Detectats)  FN (Omesos)  FP (Falses Alarmes)  Sensibilitat / Recall (%)  Precisió (%)  F1-Score (%)
    15m               236             189           47                   76                      80.08         71.32         75.45
    30m               238             146           92                  152                      61.34         48.99         54.48
    45m               243             122          121                  195                      50.21         38.49         43.57
    60m               246              86          160                  229                      34.96         27.30         30.66

Exportació de mètriques clíniques completada.
